In [24]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage, AIMessage

In [56]:
URI=  "bolt://10.1.150.105:7687"
USER = "neo4j"
PWD = "neo4jneo4j"
base_url = "http://10.1.150.105:3000/v1"
openai_api_key = "sk-0S7bV3oaznhuevN899Db0cDa20Ae4c5485Ce5862A86063C8"
model_name = 'qwen3'

graph = Neo4jGraph(url=URI, username=USER, password=PWD,enhanced_schema=True)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.AggregationSkippedNull} {category: UNRECOGNIZED} {title: The query contains an aggregation function that skips null values.} {description: null value eliminated in set function.} {position: None} for query: 'MATCH (n:`__Community__`)\nWITH collect(distinct substring(toString(n.`title`), 0, 50)) AS `title_values`,\n 

In [57]:
graph

In [58]:
text = """
Ada Lovelace was a mathematician who worked with Charles Babbage on the Analytical Engine.
She is often credited as the first computer programmer.
"""

In [59]:

from langchain_deepseek import ChatDeepSeek
llm = ChatOpenAI(base_url=base_url,openai_api_key=openai_api_key, temperature=0, model_name=model_name)
# llm = ChatDeepSeek(base_url=base_url,api_key=openai_api_key, temperature=0, model_name=model_name,streaming=False) 

llm_transformer = LLMGraphTransformer(llm=llm)

In [60]:
documents = [Document(page_content=text,metadata={"source": "https://www.jqsoft.net"})]
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Ada Lovelace', type='Person', properties={}), Node(id='Charles Babbage', type='Person', properties={}), Node(id='Analytical Engine', type='Machine', properties={})]
Relationships:[Relationship(source=Node(id='Ada Lovelace', type='Person', properties={}), target=Node(id='Charles Babbage', type='Person', properties={}), type='COLLABORATED_WITH', properties={}), Relationship(source=Node(id='Ada Lovelace', type='Person', properties={}), target=Node(id='Analytical Engine', type='Machine', properties={}), type='WORKED_ON', properties={}), Relationship(source=Node(id='Ada Lovelace', type='Person', properties={}), target=Node(id='Computer_Programmer', type='Role', properties={}), type='HAS_ROLE', properties={})]


In [61]:
graph.add_graph_documents(graph_documents, include_source=True)

In [62]:
graph.query('MATCH (n) RETURN COUNT(n) as TOTAL')

[{'TOTAL': 2793}]

In [63]:
print(graph.schema)

Node properties:
- **Document**
  - `id`: STRING Available options: ['f8ac43d46eed1a144912aa0c8c2945e5']
  - `text`: STRING Available options: [' Ada Lovelace was a mathematician who worked with ']
  - `source`: STRING Available options: ['https://www.jqsoft.net']
- **Person**
  - `id`: STRING Available options: ['Ada Lovelace', 'Charles Babbage']
- **Role**
  - `id`: STRING Available options: ['Computer_Programmer']
- **__Community__**
  - `title`: STRING Example: "Community 0"
  - `level`: INTEGER Min: 0, Max: 3
  - `community`: STRING Example: "075831db-80f3-4618-beab-221cacabd5b9"
  - `rank`: FLOAT Min: 1.0, Max: 9.5
  - `rank_explanation`: STRING Example: "该报告对智慧医院信息系统的管理标准进行了全面深入的分析，对理解并实施全面的医疗信息系统的标准、功能模"
  - `summary`: STRING Example: "该社区主要围绕智慧医院信息系统的管理标准展开，涵盖了从项目组织、管理、实施、进度、风险保障、监督控制"
  - `full_content`: STRING Example: "# 智慧医院信息系统管理标准  该社区主要围绕智慧医院信息系统的管理标准展开，涵盖了从项目组织、管理"
- **服务保障体系**
  - `id`: STRING Example: "538ad14b-8a69-468d-a680-07036bc3c5c8"
  - `description`: STRING E